In [5]:
import time
import re
import math
import collections
import pandas as pd
import chromedriver_autoinstaller

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    UnexpectedAlertPresentException,
)
from tqdm import tqdm


# ============================
# 0. 경로 설정
# ============================

# 👉 여기만 프로젝트 상황에 맞게 수정해서 사용
EXCEL_PATH = "babitalk_missing_rows.xlsx"      # 크롤링 대상 URL 엑셀
URL_COLUMN = "treatment_url"                  # 엑셀 내 URL 컬럼명
OUTPUT_CSV = "babitalk_detail_with_diagnostics_v2.csv"

MAX_ROWS = None        # None이면 전체, 숫자 넣으면 상위 N개만 테스트
SAVE_INTERVAL = 20     # 중간 저장 주기


# ============================
# 1. CSS 선택자
# ============================

# 시술명: 실제 DOM + 기존 패턴
TREATMENT_NAME_SELECTORS = [
    "h1.text-label-common_5",
    "h2.text-label-skinTreatment_2",
    "h2.text-label-plasticsurgery_2",
    "#content-body h1",
    "#content-body h2",
]

# 병원명
HOSPITAL_NAME_SELECTORS = [
    "p.text-label-common_4.cursor-pointer",
    "#hospital h4",
    "#hospital h3",
    "#hospital p",
    "#hospital span",
]

# 평점 / 후기
RATING_SELECTOR = r"div.flex.items-center.gap-\[10px\] h5"
REVIEW_COUNT_SELECTOR = r"div.flex.items-center.gap-\[10px\] button"

# 가격(정가)
ORIGINAL_PRICE_SELECTORS = [
    "h6.text-label-skinTreatment_2",
    "h6.text-label-plasticsurgery_2",
    "h2 span",
]

# 할인가
SALE_PRICE_SELECTOR = r"h2.flex.items-center.gap-\[2px\]"

# 할인율
DISCOUNT_RATE_SELECTORS = [
    "h2.text-label-skinTreatment_2",
    "h2.text-label-plasticsurgery_2",
]

# 가격 텍스트가 실제로 들어가는 span (스크린샷 기준)
PRICE_TEXT_SELECTORS = [
    "span[class*='text-[24px]'][class*='font-bold']",
    "h2.flex.items-center.gap-[2px]",
]

# 이벤트/부작용
EVENT_DESC_SELECTOR = "#detail div.bg-white div div"
EVENT_PERIOD_SELECTOR = "#detail .bg-neutral-100 div:nth-child(1) .text-label-common_3"
EVENT_TARGET_SELECTOR = "#detail .bg-neutral-100 div:nth-child(2) .text-label-common_3"
SIDE_EFFECT_SELECTOR  = "#detail .bg-neutral-100 div:nth-child(3) .text-label-common_3"

# 병원 주소
HOSPITAL_ADDRESS_SELECTORS = [
    r"#hospital div.flex.flex-col.justify-start.items-start.gap-\[4px\] h5:nth-child(3)",
    "#hospital h5:nth-child(3)",
    "#hospital h5",
    "#hospital p",
    "#hospital span",
]

# "시술 정보 더보기" 버튼
MORE_DETAIL_BTN_SELECTOR = "#btn_more_detail"


# ============================
# 2. 공용 함수
# ============================

def init_driver():
    chromedriver_autoinstaller.install()
    options = Options()
    options.add_argument("--start-maximized")
    # options.add_argument("--headless")  # 서버에서 돌릴 때 켜도 됨
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    driver = webdriver.Chrome(options=options)
    return driver


def safe_text(driver, selector):
    try:
        el = driver.find_element(By.CSS_SELECTOR, selector)
        txt = el.text.strip()
        return txt if txt else None
    except Exception:
        return None


def first_non_empty(driver, selectors):
    for s in selectors:
        txt = safe_text(driver, s)
        if txt:
            return txt
    return None


def extract_number(text):
    if not text:
        return None
    num = re.sub(r"[^\d]", "", text)
    return int(num) if num else None


def handle_alert_if_exists(driver):
    try:
        alert = driver.switch_to.alert
        print(f"[알럿] {alert.text}")
        alert.accept()
        return True
    except Exception:
        return False


def click_more_detail_if_exists(driver):
    try:
        btns = driver.find_elements(By.CSS_SELECTOR, MORE_DETAIL_BTN_SELECTOR)
        if not btns:
            return False

        btn = btns[0]
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(0.5)

        handle_alert_if_exists(driver)
        return True
    except UnexpectedAlertPresentException:
        handle_alert_if_exists(driver)
        return False
    except Exception:
        return False


def parse_discount_rate(driver):
    for sel in DISCOUNT_RATE_SELECTORS:
        txt = safe_text(driver, sel)
        if txt and "%" in txt:
            return txt.strip()

    try:
        h2s = driver.find_elements(By.CSS_SELECTOR, "h2")
        for h in h2s:
            t = h.text.strip()
            if "%" in t:
                return t
    except Exception:
        pass

    return None


def parse_price_with_fallback(driver, original_price_text, sale_price_text):
    """
    가격이 비어있을 때 fallback으로 #detail/body에서 '숫자+원' 텍스트 스캔.
    최대값 = 정가, 최소값 = 할인가(있다면)로 추정.
    """
    if original_price_text and sale_price_text:
        return original_price_text, sale_price_text

    try:
        try:
            container = driver.find_element(By.CSS_SELECTOR, "#detail")
        except Exception:
            container = driver.find_element(By.TAG_NAME, "body")

        candidates = []
        texts = container.find_elements(By.CSS_SELECTOR, "h2, h3, span")
        for el in texts:
            t = el.text.strip()
            if not t:
                continue
            if "원" in t and re.search(r"\d", t):
                if len(t) > 30:
                    continue
                candidates.append(t)

        if not candidates:
            return original_price_text, sale_price_text

        candidates = list(dict.fromkeys(candidates))

        priced = []
        for c in candidates:
            n = extract_number(c)
            if n:
                priced.append((n, c))

        if not priced:
            return original_price_text, sale_price_text

        priced.sort(key=lambda x: x[0])
        min_v, min_txt = priced[0]
        max_v, max_txt = priced[-1]

        if not original_price_text and not sale_price_text:
            if len(priced) == 1:
                original_price_text = priced[0][1]
            else:
                original_price_text = max_txt
                sale_price_text = min_txt
        else:
            if not original_price_text:
                original_price_text = max_txt
            if not sale_price_text and len(priced) > 1:
                sale_price_text = min_txt

        return original_price_text, sale_price_text

    except Exception:
        return original_price_text, sale_price_text


def wait_for_basic_dom(driver, timeout=15):
    """
    기본 DOM(#content-body 또는 h1/h2)이 뜰 때까지 대기
    """
    WebDriverWait(driver, timeout).until(
        EC.any_of(
            EC.presence_of_element_located((By.CSS_SELECTOR, "#content-body")),
            EC.presence_of_element_located((By.CSS_SELECTOR, "h1")),
            EC.presence_of_element_located((By.CSS_SELECTOR, "h2")),
        )
    )


def wait_for_text_in_selectors(driver, selectors, timeout=15):
    """
    selectors 중 하나라도 '텍스트가 비어있지 않은 상태'가 될 때까지 대기
    (SPA 비동기 로딩 대응용)
    """
    def _has_non_empty_text(d):
        for sel in selectors:
            try:
                els = d.find_elements(By.CSS_SELECTOR, sel)
                for el in els:
                    if el.text and el.text.strip():
                        return True
            except Exception:
                continue
        return False

    WebDriverWait(driver, timeout).until(_has_non_empty_text)


def diagnose_missing(row_dict, exclude_keys=None):
    """
    row_dict에서 None / NaN 인 컬럼 이름 리스트 반환
    """
    if exclude_keys is None:
        exclude_keys = []

    missing_cols = []
    for k, v in row_dict.items():
        if k in exclude_keys:
            continue
        if v is None:
            missing_cols.append(k)
        elif isinstance(v, float) and math.isnan(v):
            missing_cols.append(k)

    return missing_cols


# ============================
# 3. 단일 페이지 크롤링
# ============================

def crawl_treatment_page(driver, url: str) -> dict:
    driver.get(url)

    # 기본 DOM + 텍스트 로딩 대기
    try:
        wait_for_basic_dom(driver, timeout=20)
        wait_for_text_in_selectors(
            driver,
            selectors=TREATMENT_NAME_SELECTORS + PRICE_TEXT_SELECTORS,
            timeout=20,
        )
        crawl_status = "ok"
    except TimeoutException:
        # 텍스트 로딩 조건을 만족 못해도, 일단 수집은 시도
        print(f"❗ [Timeout] 텍스트까지 로딩 안 됨 → {url}")
        crawl_status = "timeout"

    time.sleep(0.3)

    # 필드 수집
    treatment_name   = first_non_empty(driver, TREATMENT_NAME_SELECTORS)
    hospital_name    = first_non_empty(driver, HOSPITAL_NAME_SELECTORS)
    hospital_address = first_non_empty(driver, HOSPITAL_ADDRESS_SELECTORS)

    rating_text = safe_text(driver, RATING_SELECTOR)
    try:
        rating = float(rating_text) if rating_text else None
    except ValueError:
        rating = None

    review_raw = safe_text(driver, REVIEW_COUNT_SELECTOR)
    review_num = extract_number(review_raw)
    review_count = f"{review_num}개" if review_num is not None else None

    original_price = first_non_empty(driver, ORIGINAL_PRICE_SELECTORS)
    discount_rate  = parse_discount_rate(driver)
    sale_price     = safe_text(driver, SALE_PRICE_SELECTOR)

    original_price, sale_price = parse_price_with_fallback(
        driver,
        original_price_text=original_price,
        sale_price_text=sale_price,
    )

    click_more_detail_if_exists(driver)
    time.sleep(0.3)

    event_desc   = safe_text(driver, EVENT_DESC_SELECTOR)
    event_period = safe_text(driver, EVENT_PERIOD_SELECTOR)
    event_target = safe_text(driver, EVENT_TARGET_SELECTOR)
    side_effect  = safe_text(driver, SIDE_EFFECT_SELECTOR)

    row = {
        "treatment_url": url,
        "crawl_status": crawl_status,  # ok / timeout / error:...
        "treatment_name": treatment_name,
        "rating": rating,
        "review_count": review_count,
        "original_price_text": original_price,
        "discount_rate_text": discount_rate,
        "sale_price_text": sale_price,
        "event_description": event_desc,
        "event_period": event_period,
        "event_target": event_target,
        "side_effect_info": side_effect,
        "hospital_name": hospital_name,
        "hospital_address": hospital_address,
    }

    # 어떤 컬럼이 비었는지 진단
    missing_cols = diagnose_missing(
        row,
        exclude_keys=["treatment_url", "crawl_status"]
    )
    row["missing_count"] = len(missing_cols)
    row["missing_cols"] = ", ".join(missing_cols) if missing_cols else ""

    # 콘솔 출력
    if len(missing_cols) == 0:
        print(f"✅ [완전수집] url={url}")
    elif len(missing_cols) <= 3:
        print(f"⚠️ [일부 누락] url={url} → {missing_cols}")
    else:
        print(f"🚨 [다수 누락] url={url} → {missing_cols} (status={crawl_status})")

    return row


# ============================
# 4. 메인 – 전체 URL 크롤링 + 중간저장
# ============================

def main():
    df = pd.read_excel(EXCEL_PATH)

    if URL_COLUMN not in df.columns:
        raise ValueError(f"엑셀에 '{URL_COLUMN}' 컬럼이 없습니다.")

    urls = (
        df[URL_COLUMN]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    if MAX_ROWS is not None:
        urls = urls[:MAX_ROWS]

    print(f"크롤링 대상 URL 수: {len(urls)}")

    driver = init_driver()
    results = []

    try:
        for i, url in enumerate(
            tqdm(urls, desc="바비톡 시술 크롤링 (진단 포함)", unit="건"),
            start=1
        ):
            try:
                row = crawl_treatment_page(driver, url)
                results.append(row)
            except Exception as e:
                print(f"❗ [에러] idx={i}, url={url} / {e}")
                results.append({
                    "treatment_url": url,
                    "crawl_status": f"error: {e}",
                    "missing_count": None,
                    "missing_cols": None,
                })

            # 중간 저장
            if i % SAVE_INTERVAL == 0:
                tmp_df = pd.DataFrame(results)
                tmp_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
                print(f"💾 중간 저장({i}건) → {OUTPUT_CSV}")

            time.sleep(1)   # 서버 부하 방지
    finally:
        driver.quit()

    pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ 크롤링 완료 – 총 {len(results)}건 저장 → {OUTPUT_CSV}")


if __name__ == "__main__":
    main()


크롤링 대상 URL 수: 1525


바비톡 시술 크롤링 (진단 포함):   0%|          | 0/1525 [00:00<?, ?건/s]

⚠️ [일부 누락] url=https://web.babitalk.com/events/66930 → ['rating', 'review_count', 'discount_rate_text']


바비톡 시술 크롤링 (진단 포함):   0%|          | 1/1525 [00:04<1:57:37,  4.63s/건]

⚠️ [일부 누락] url=https://web.babitalk.com/events/59582 → ['discount_rate_text']


...

✅ [완전수집] url=https://web.babitalk.com/events/45541


바비톡 시술 크롤링 (진단 포함): 100%|█████████▉| 1524/1525 [1:52:10<00:03,  3.36s/건]

✅ [완전수집] url=https://web.babitalk.com/events/34588


바비톡 시술 크롤링 (진단 포함): 100%|██████████| 1525/1525 [1:52:14<00:00,  4.42s/건]



✅ 크롤링 완료 – 총 1525건 저장 → babitalk_detail_with_diagnostics_v2.csv


In [2]:
!pip install chromedriver-autoinstaller